In [1]:
import sys, gc
from pn.feature_extraction import *
from pn.sig_proc import * 
from intanutil import intan_rhd as intan 

In [2]:
'''
TO DO: This should probably get folded into the 'Config' class that's currently being used for the 3DCNN? 
'''
PARAMETERS = {
   
    # parameters loading raw data
    'file_path': "D:\\Becca\\Data\\PN\\NSR-101-002\\ICU-Session6\\Train1-center-out-vert-easy-FIH0058\\",  
    'ephys_alignment': False,
    'imp_file': "D:\\Becca\\Data\\PN\\NSR-101-002\\ICU-Session6\\impedance1_12_FIH0058_ImpedanceMeasurement1kHz.csv", 
    'imp_thresh_ohm': 4_000_000,
    #'feature_file': "D:/Becca/Data/CC01_features/F1/ccCenterOut_10142022_S03_F1_50_10.pkl",  # which file to read in data from (converted to 
                                                         #features w feat_extraction because it takes forever)
    # Parameters for cursor motion filtering 
     'cutoff_frequency': 6,  # Hz
    
    # Parameters for separating into trials
    'task_split': 'trigger', # 'velocity'
    #'settle_time_ms':       500,
    #'motion_filer_order':   10, 
    #'motion_filter_highcut':8, 


    # Parameters for feature / label  extraction 
    #'feature_type': "fft",
    'window_len': 250, # window length in ms
    'window_stride': 10, #window stride in ms 
    #'bucket_count': 8,  # number of buckets to split angles into (classes will be this + a rest class)
         
}

In [3]:
'''
This returns a dictionary that countains the various components from the data block

Experiment Parameters:
'fs':                           intan sampling rate
'ephys_fs':                     open ephys sampling rate


From the Intan:
'amplifier_data':  intan amplifier data 
'amplifier_timing': intan timing (not sure if this is actually useful),
'trigger_data':   external sync signal, arduino or otherwise
'trigger_timing':   
From OpenEphys(Task running on Computer). This data is asynchronous. XY is cursor position, contains X then Y separately. Vx, Vy velocities
data is data packets
timestamps is the time in nanoseconds that respective packets were recieved, starting from 0 on recording start
wall_clock is the time on the PC clock that the recording was started (for aligning Ephys data)

'xy_timestamps': 
'xy_wall_clock':              
'x_data':                      
'y_data':                      
'vx_timestamps':               
'vx_wall_clock':              
'vx_data':                    
'vy_timestamps':             
'vy_wall_clock':              
'vy_data':                      

If there was an arduino used to sync OpenEphys and the intan, these are used to do the time alignment: 
         
'ard_data':                     ard_data,
'ard_timestamps':               ard_timestamps,
'ard_wall_clock':               ard_wall_clock,

'''

block_data = load_pn_ccout_from_path(PARAMETERS["file_path"])
gc.collect()

Reading the dat file...
Reading the amplifier data as an signed integer array ..
ISSUE WITH ANALOGIN
This is the board mode: 13
Using the raw values of the data, i.e. not scaling to voltage values. Not doing (x - 2**15) * 0.195. Just x.
Convert to floating point values.
Conversion to float: Execution time: 9.590726852416992 seconds
Complete converting to floating point values.
Check for gaps in timestamps.
Checking for gaps in timestamps.
Check for gap: Execution time: 0.012424230575561523 seconds
Complete checking for gaps in timestamps.
Scale time steps.
scale time stamps: Execution time: 0.08303213119506836 seconds
Complete parse_raw_data().
No Trigger Found
No Ephys Arduino Data Found


86

In [5]:
''' Set Values to 0 for high impedance channels. This should maybe be 'NaN' instead of 0 ''' 
zero_high_imp_channels(PARAMETERS, block_data)


In [6]:
''' Ephys Data Alignment
NSR102-005 uses a different method. I don't know how to align for NSR101-XXX, we need to ask
'''
if PARAMETERS['ephys_alignment'] and block_data['ard_data'] is not None:
    align_intan_ephys(block_data)
    # This probably neds even more options.
    filter_motion_data(PARAMETERS, block_data)
    gc.collect()
gc.collect()

0

In [8]:
def detect_rising_edge(sequence, threshold, cooldown, mode='index_output'):
    step_indices = []
    in_cooldown = 0
    nSize = len(sequence)
    seq_diff = np.diff(sequence)

    thres_times = np.argwhere(seq_diff > threshold)[:,0]
    step_indices.append(thres_times[0])
    for t in thres_times[1:]:
        if t - step_indices[-1] > cooldown:
            step_indices.append(t)

    return np.array(step_indices)

In [12]:
trig = detect_rising_edge(block_data['trigger_data'], threshold=0.5, cooldown=500, mode='index_output')
print(trig.shape)

TypeError: object of type 'NoneType' has no len()

In [ ]:
'''
Can split into trials based on velocity profiles, or trigger if available

Each trial stores x,y neural data for the active part of the trial, and the ecog data that will be used as a baseline (for a running-baseline implimentation)
Also stores a dictionary of different features and assosciated labels for that Trial, to be populated.
'''
match PARAMETERS['task_split']:
    case 'velocity':
        trial_list = separate_trials_velocity(block_data)
    case 'trigger':
        # need to work on this
    case _: print("INVALID SPLITTING METHOD")

In [ ]:
'''
You will probably want to extract features one trial at a time because otherwise the filters eat up a large amount of memory
'''
for task in task_list: 
    extract_ssep_basic(PARAMETERS, task)